In [6]:
import numpy as np
import librosa
import os
import csv
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

# =========================
# PARAMETERS
# =========================
B = 12
fmin = 32.7
hop_length = 512
max_len = 200
n_components = 100

audio_root = "dataset_full"
feature_root = "features_full_cqt"

# Subfolders
raw_dir = os.path.join(feature_root, "raw")
flat_dir = os.path.join(feature_root, "flattened")
pca_dir = os.path.join(feature_root, "pca")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(flat_dir, exist_ok=True)
os.makedirs(pca_dir, exist_ok=True)

# =========================
# FEATURE FUNCTION
# =========================
def compute_cqt_full(filename):
    x, fs = librosa.load(filename, sr=None)

    X_cq = librosa.cqt(
        x,
        sr=fs,
        hop_length=hop_length,
        fmin=fmin,
        bins_per_octave=B
    )

    X_mag = np.abs(X_cq)
    X_db = librosa.amplitude_to_db(X_mag)

    # Normalize
    X_db = (X_db - np.mean(X_db)) / (np.std(X_db) + 1e-6)

    # Fix length
    if X_db.shape[1] < max_len:
        pad = max_len - X_db.shape[1]
        X_db = np.pad(X_db, ((0, 0), (0, pad)), mode='constant')
    else:
        X_db = X_db[:, :max_len]

    return X_db

# =========================
# PROCESS ALL FILES
# =========================
all_flattened = []
file_keys = []
labels = []

processed = 0

for label in os.listdir(audio_root):
    class_dir = os.path.join(audio_root, label)

    if not os.path.isdir(class_dir):
        continue

    for file in os.listdir(class_dir):
        if not file.endswith(".wav"):
            continue

        sample_key = f"{label}_{file.replace('.wav', '')}"
        audio_path = os.path.join(class_dir, file)

        try:
            cqt_matrix = compute_cqt_full(audio_path)

            # -------- RAW --------
            raw_path = os.path.join(raw_dir, f"{sample_key}.npy")
            np.save(raw_path, cqt_matrix)

            # -------- FLATTENED --------
            flat = cqt_matrix.flatten()
            flat_path = os.path.join(flat_dir, f"{sample_key}.npy")
            np.save(flat_path, flat)

            # Store for PCA
            all_flattened.append(flat)
            file_keys.append(sample_key)
            labels.append(label)

            processed += 1

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")

print(f"Processed for raw & flattened: {processed}")

# =========================
# PCA COMPUTATION
# =========================
print("Running PCA...")

X_flat = np.array(all_flattened)

# Safety check
n_components = min(n_components, X_flat.shape[0])

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_flat)

# =========================
# SAVE PCA FEATURES
# =========================
for i, key in enumerate(file_keys):
    pca_path = os.path.join(pca_dir, f"{key}.npy")
    np.save(pca_path, X_pca[i])

# =========================
# SAVE LABELS AS CSV
# =========================
le = LabelEncoder()
y = le.fit_transform(labels)

csv_path = os.path.join(pca_dir, "labels.csv")

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    # header
    writer.writerow(["file_key", "label", "label_encoded"])
    
    # rows
    for i in range(len(file_keys)):
        writer.writerow([file_keys[i], labels[i], y[i]])

print("PCA features and labels CSV saved.")


# =========================
# PARAMETERS
# =========================
n_mfcc = 20
hop_length = 512
n_components = 20   # MFCC already small → no need 100

audio_root = "dataset_full"
feature_root = "features_full_mfcc"

# Subfolders
raw_dir = os.path.join(feature_root, "raw")
flat_dir = os.path.join(feature_root, "flattened")
pca_dir = os.path.join(feature_root, "pca")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(flat_dir, exist_ok=True)
os.makedirs(pca_dir, exist_ok=True)

# =========================
# FEATURE FUNCTION
# =========================
def compute_mfcc(filename):
    x, fs = librosa.load(filename, sr=None)
    
    X_mfcc = librosa.feature.mfcc(
        y=x,
        sr=fs,
        n_mfcc=n_mfcc,
        hop_length=hop_length
    )
    
    # Mean over time (your method)
    features = np.mean(X_mfcc, axis=1)

    # Normalize (IMPORTANT for SVM)
    features = (features - np.mean(features)) / (np.std(features) + 1e-6)
    
    return features

# =========================
# PROCESS DATASET
# =========================
all_features = []
file_keys = []
labels = []

processed = 0

for label in os.listdir(audio_root):
    class_dir = os.path.join(audio_root, label)

    if not os.path.isdir(class_dir):
        continue

    for file in os.listdir(class_dir):
        if not file.endswith(".wav"):
            continue

        sample_key = f"{label}_{file.replace('.wav', '')}"
        audio_path = os.path.join(class_dir, file)

        try:
            features = compute_mfcc(audio_path)

            # -------- RAW --------
            raw_path = os.path.join(raw_dir, f"{sample_key}.npy")
            np.save(raw_path, features)

            # -------- FLATTENED --------
            flat_path = os.path.join(flat_dir, f"{sample_key}.npy")
            np.save(flat_path, features)

            all_features.append(features)
            file_keys.append(sample_key)
            labels.append(label)

            processed += 1

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")

print(f"Processed: {processed}")

# =========================
# PCA (optional but for fairness)
# =========================
X = np.array(all_features)

n_components = min(n_components, X.shape[0])

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X)

# Save PCA features
for i, key in enumerate(file_keys):
    np.save(os.path.join(pca_dir, f"{key}.npy"), X_pca[i])

# =========================
# SAVE LABELS CSV
# =========================
le = LabelEncoder()
y = le.fit_transform(labels)

csv_path = os.path.join(pca_dir, "labels.csv")

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["file_key", "label", "label_encoded"])
    
    for i in range(len(file_keys)):
        writer.writerow([file_keys[i], labels[i], y[i]])

print("MFCC features and labels saved.")

Processed for raw & flattened: 6705
Running PCA...
PCA features and labels CSV saved.
Processed: 6705
MFCC features and labels saved.


In [2]:
import numpy as np
import librosa
import os
import csv
from sklearn.preprocessing import LabelEncoder

# =========================
# PATHS
# =========================
audio_root = "dataset_full"
feature_root = "features_full_summary"

os.makedirs(feature_root, exist_ok=True)

# =========================
# FEATURE FUNCTION
# =========================
def extract_summary_features(filename):
    x, sr = librosa.load(filename, sr=None)

    # -------- TIME DOMAIN --------
    zcr = np.mean(librosa.feature.zero_crossing_rate(x))
    rms = np.mean(librosa.feature.rms(y=x))

    # -------- SPECTRAL --------
    centroid = np.mean(librosa.feature.spectral_centroid(y=x, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=x, sr=sr))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=x, sr=sr))

    # Final feature vector (5 features)
    features = np.array([
        zcr,
        rms,
        centroid,
        bandwidth,
        rolloff
    ])

    return features

# =========================
# PROCESS DATASET
# =========================
file_keys = []
labels = []

processed = 0

for label in os.listdir(audio_root):
    class_dir = os.path.join(audio_root, label)

    if not os.path.isdir(class_dir):
        continue

    for file in os.listdir(class_dir):
        if not file.endswith(".wav"):
            continue

        sample_key = f"{label}_{file.replace('.wav', '')}"
        audio_path = os.path.join(class_dir, file)
        save_path = os.path.join(feature_root, f"{sample_key}.npy")

        try:
            features = extract_summary_features(audio_path)

            np.save(save_path, features)

            file_keys.append(sample_key)
            labels.append(label)

            processed += 1

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")

print(f"Processed: {processed}")

# =========================
# SAVE LABELS CSV
# =========================
le = LabelEncoder()
y = le.fit_transform(labels)

csv_path = os.path.join(feature_root, "labels.csv")

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["file_key", "label", "label_encoded"])

    for i in range(len(file_keys)):
        writer.writerow([file_keys[i], labels[i], y[i]])

print("Summary features and labels saved.")

Processed: 6705
Summary features and labels saved.


In [3]:
import numpy as np
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =========================
# LOAD LABELS
# =========================
df = pd.read_csv("features_full_summary/labels.csv")
label_names = sorted(df["label"].unique())

file_keys = df["file_key"].values
y = df["label_encoded"].values

# =========================
# CONFIG
# =========================
summary_dir = "features_full_summary"
mfcc_dir = "features_full_mfcc/pca"
cqt_dir = "features_full_cqt/pca"

# =========================
# BUILD DATASETS
# =========================
X_mfcc = []
X_cqt = []
X_all = []
y_filtered = []

for i, key in enumerate(file_keys):
    summary_path = os.path.join(summary_dir, f"{key}.npy")
    mfcc_path = os.path.join(mfcc_dir, f"{key}.npy")
    cqt_path = os.path.join(cqt_dir, f"{key}.npy")

    if not (os.path.exists(summary_path) and os.path.exists(mfcc_path) and os.path.exists(cqt_path)):
        continue

    summary_feat = np.load(summary_path)
    mfcc_feat = np.load(mfcc_path)
    cqt_feat = np.load(cqt_path)

    # -------------------------
    # Normalize EACH feature group
    # -------------------------
    summary_feat = (summary_feat - np.mean(summary_feat)) / (np.std(summary_feat) + 1e-6)
    mfcc_feat = (mfcc_feat - np.mean(mfcc_feat)) / (np.std(mfcc_feat) + 1e-6)
    cqt_feat = (cqt_feat - np.mean(cqt_feat)) / (np.std(cqt_feat) + 1e-6)

    # -------------------------
    # COMBINATIONS
    # -------------------------
    combined_mfcc = np.concatenate([summary_feat, mfcc_feat])
    combined_cqt = np.concatenate([summary_feat, cqt_feat])
    combined_all = np.concatenate([summary_feat, mfcc_feat, cqt_feat])

    X_mfcc.append(combined_mfcc)
    X_cqt.append(combined_cqt)
    X_all.append(combined_all)
    y_filtered.append(y[i])

X_mfcc = np.array(X_mfcc)
X_cqt = np.array(X_cqt)
X_all = np.array(X_all)
y_filtered = np.array(y_filtered)

print("MFCC shape:", X_mfcc.shape)
print("CQT  shape:", X_cqt.shape)
print("ALL  shape:", X_all.shape)

# =========================
# TRAIN / TEST SPLIT
# =========================
X_train_mfcc, X_test_mfcc, y_train, y_test = train_test_split(
    X_mfcc, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_cqt, X_test_cqt, _, _ = train_test_split(
    X_cqt, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_all, X_test_all, _, _ = train_test_split(
    X_all, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

# =========================
# SVM FUNCTION
# =========================
from sklearn.metrics import classification_report

def train_and_evaluate(X_train, X_test, y_train, y_test, name, label_names):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # -------------------------
    # OVERALL METRICS
    # -------------------------
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")

    # -------------------------
    # CLASSIFICATION REPORT
    # -------------------------
    print("\n--- Per-Class Metrics ---")
    print(classification_report(y_test, y_pred, target_names=label_names))

    # -------------------------
    # PER-CLASS ACCURACY (explicit)
    # -------------------------
    print("\n--- Accuracy per Instrument ---")
    for i, label in enumerate(label_names):
        idx = (y_test == i)
        correct = np.sum(y_pred[idx] == y_test[idx])
        total = np.sum(idx)

        acc_i = correct / total if total > 0 else 0
        print(f"{label}: {acc_i:.4f} ({correct}/{total})")

# =========================
# RUN EXPERIMENTS
# =========================
train_and_evaluate(X_train_mfcc, X_test_mfcc, y_train, y_test,
                   "SUMMARY + MFCC (PCA)", label_names)

train_and_evaluate(X_train_cqt, X_test_cqt, y_train, y_test,
                   "SUMMARY + CQT (PCA)", label_names)

train_and_evaluate(X_train_all, X_test_all, y_train, y_test,
                   "SUMMARY + MFCC + CQT (PCA)", label_names)

MFCC shape: (6705, 25)
CQT  shape: (6705, 105)
ALL  shape: (6705, 125)

===== SUMMARY + MFCC (PCA) =====
Accuracy : 0.6585
F1 Score : 0.6576
Precision: 0.6620
Recall   : 0.6585

--- Per-Class Metrics ---
              precision    recall  f1-score   support

         cel       0.65      0.68      0.67        78
         cla       0.58      0.64      0.61       101
         flu       0.58      0.59      0.58        90
         gac       0.61      0.76      0.68       127
         gel       0.65      0.70      0.67       152
         org       0.72      0.76      0.74       136
         pia       0.73      0.64      0.68       144
         sax       0.62      0.56      0.59       125
         tru       0.74      0.63      0.68       116
         vio       0.67      0.53      0.59       116
         voi       0.68      0.70      0.69       156

    accuracy                           0.66      1341
   macro avg       0.66      0.65      0.65      1341
weighted avg       0.66      0.66     

In [4]:
# Add channel dimension for CNN
X_mfcc_cnn = X_mfcc[..., np.newaxis]
X_cqt_cnn = X_cqt[..., np.newaxis]
X_all_cnn = X_all[..., np.newaxis]

print("MFCC CNN shape:", X_mfcc_cnn.shape)
print("CQT  CNN shape:", X_cqt_cnn.shape)
print("ALL  CNN shape:", X_all_cnn.shape)

from sklearn.model_selection import train_test_split

X_train_mfcc, X_test_mfcc, y_train, y_test = train_test_split(
    X_mfcc_cnn, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_cqt, X_test_cqt, _, _ = train_test_split(
    X_cqt_cnn, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

X_train_all, X_test_all, _, _ = train_test_split(
    X_all_cnn, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

from tensorflow.keras import layers, models

def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv1D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling1D(2),

        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),

        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

from sklearn.metrics import classification_report

def train_and_evaluate_cnn(X_train, X_test, y_train, y_test, name, label_names):
    model = build_cnn(X_train.shape[1:], len(label_names))

    print(f"\nTraining {name}...")

    model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # Evaluate
    loss, acc = model.evaluate(X_test, y_test, verbose=0)

    y_pred = np.argmax(model.predict(X_test), axis=1)

    print(f"\n===== {name} (CNN) =====")
    print(f"Accuracy: {acc:.4f}")

    print("\n--- Per-Class Metrics ---")
    print(classification_report(y_test, y_pred, target_names=label_names))

label_names = sorted(df["label"].unique())

train_and_evaluate_cnn(X_train_mfcc, X_test_mfcc, y_train, y_test,
                       "SUMMARY + MFCC", label_names)

train_and_evaluate_cnn(X_train_cqt, X_test_cqt, y_train, y_test,
                       "SUMMARY + CQT", label_names)

train_and_evaluate_cnn(X_train_all, X_test_all, y_train, y_test,
                       "SUMMARY + MFCC + CQT", label_names)

MFCC CNN shape: (6705, 25, 1)
CQT  CNN shape: (6705, 105, 1)
ALL  CNN shape: (6705, 125, 1)


C:\Users\windows 10\tf_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training SUMMARY + MFCC...
Epoch 1/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2345 - loss: 2.1674 - val_accuracy: 0.3177 - val_loss: 2.0400
Epoch 2/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3111 - loss: 2.0052 - val_accuracy: 0.3184 - val_loss: 1.9364
Epoch 3/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3494 - loss: 1.9036 - val_accuracy: 0.3647 - val_loss: 1.8596
Epoch 4/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3842 - loss: 1.8168 - val_accuracy: 0.3818 - val_loss: 1.8193
Epoch 5/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4068 - loss: 1.7456 - val_accuracy: 0.4131 - val_loss: 1.7558
Epoch 6/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4370 - loss: 1.6785 - val_accuracy: 0.4198 - val_loss: 1.7145
Epoch 7/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4528 - loss: 1.6226 - val_accuracy: 0.4280 - val_loss: 1.6875
Epoch 8/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.48

C:\Users\windows 10\tf_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


168/168 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.2008 - loss: 2.2341 - val_accuracy: 0.2662 - val_loss: 2.1131
Epoch 2/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2783 - loss: 2.0596 - val_accuracy: 0.2901 - val_loss: 2.0382
Epoch 3/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3221 - loss: 1.9438 - val_accuracy: 0.3005 - val_loss: 1.9822
Epoch 4/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3691 - loss: 1.8312 - val_accuracy: 0.3423 - val_loss: 1.9498
Epoch 5/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4135 - loss: 1.7070 - val_accuracy: 0.3445 - val_loss: 1.9333
Epoch 6/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4586 - loss: 1.5596 - val_accuracy: 0.3609 - val_loss: 1.8957
Epoch 7/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5168 - loss: 1.4150 - val_accuracy: 0.3736 - val_loss: 1.9229
Epoch 8/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5749 - loss: 1.2483 - val_accuracy: 0.3

C:\Users\windows 10\tf_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


168/168 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.2405 - loss: 2.1509 - val_accuracy: 0.3356 - val_loss: 1.9676
Epoch 2/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3374 - loss: 1.9155 - val_accuracy: 0.3699 - val_loss: 1.8544
Epoch 3/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3999 - loss: 1.7655 - val_accuracy: 0.3990 - val_loss: 1.7815
Epoch 4/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4497 - loss: 1.6160 - val_accuracy: 0.4288 - val_loss: 1.7103
Epoch 5/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5026 - loss: 1.4771 - val_accuracy: 0.4422 - val_loss: 1.6756
Epoch 6/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5569 - loss: 1.3064 - val_accuracy: 0.4474 - val_loss: 1.6730
Epoch 7/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6201 - loss: 1.1392 - val_accuracy: 0.4638 - val_loss: 1.7090
Epoch 8/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6613 - loss: 1.0034 - val_accuracy: 0.464